# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. All dataset elements are referenced by their Croissant schema `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # access as object
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` fields.

The FAIR² dataset contains multiple record sets describing analyses of protein abundance, modifications, molecular weights, etc.

Let's enumerate the record sets and their field `@id`s using the metadata object.

In [ ]:
# List available record sets and their fields (referenced by @id)
record_sets = metadata.recordSet
if not record_sets:
    print("No record sets are directly listed in top-level metadata. Try reading dataset.files for dataset contents.")

# However, most Croissant datasets store recordSets in schema:distribution (@id).
record_set_ids = []

print("Available Record Sets:")
for record_set in dataset.record_sets():
    print(f"- @id: {record_set['@id']}, name: {record_set.get('name', 'N/A')}")
    field_ids = [f['@id'] for f in record_set.get('field', [])]
    print(f"  Fields: {field_ids}")
    record_set_ids.append(record_set['@id'])

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Here, we extract the records for each record set present in the dataset.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nProcessing record set {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Fields (columns) for record set {record_set_id}:")
            print(df.columns.tolist())
            display(df.head())
        else:
            print(f"No records found for record set {record_set_id}.")
    except Exception as e:
        print(f"  Error loading record set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Let's process one of the available record sets. We'll:

- Select a numeric field (e.g., "MW_(kDa)" for molecular weight if present),
- Filter by threshold,
- Normalize and group by protein description if available.

_**Use the correct `@id` fields for columns/fields.**_

In [ ]:
# For demonstration, select the first record set with a numeric field
chosen_record_set_id = None
chosen_numeric_field = None
for rid, df in dataframes.items():
    numeric_candidates = [c for c in df.columns if df[c].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[c])]
    if numeric_candidates:
        chosen_record_set_id = rid
        chosen_numeric_field = numeric_candidates[0]
        break

if chosen_record_set_id is None:
    raise RuntimeError('No appropriate numeric fields found for EDA.')

print(f"Analyzing record set: {chosen_record_set_id} with numeric field: {chosen_numeric_field}")
threshold = 10
filtered_df = dataframes[chosen_record_set_id][dataframes[chosen_record_set_id][chosen_numeric_field] > threshold]
print(f"Filtered records with {chosen_numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{chosen_numeric_field}_normalized"] = (
    filtered_df[chosen_numeric_field] - filtered_df[chosen_numeric_field].mean()
) / (filtered_df[chosen_numeric_field].std() + 1e-9)
print(f"Normalized {chosen_numeric_field} (mean=0, std=1):")
display(filtered_df[[chosen_numeric_field, f"{chosen_numeric_field}_normalized"]].head())

# Try grouping by a categorical/text field if present
group_field = None
for c in filtered_df.columns:
    if c != chosen_numeric_field and filtered_df[c].dtype == object:
        group_field = c
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[chosen_numeric_field].mean().reset_index()
    print(f"Grouped mean {chosen_numeric_field} by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Let's plot the distribution of the selected numeric field and, if grouped, the group means.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
filtered_df[chosen_numeric_field].hist(bins=30, alpha=0.7)
plt.xlabel(chosen_numeric_field)
plt.ylabel('Frequency')
plt.title(f'Distribution of {chosen_numeric_field} (>{threshold})')
plt.show()

# If we performed grouping above, also plot group means
if group_field and 'grouped_df' in locals():
    plt.figure(figsize=(10, 5))
    grouped_df.plot.bar(x=group_field, y=chosen_numeric_field, legend=False)
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {chosen_numeric_field}')
    plt.title(f'Mean {chosen_numeric_field} per {group_field}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- We loaded the FAIR² mass spectrometry dataset using its Croissant schema and explored record sets via their `@id` fields.
- We demonstrated basic filtering, normalization, grouping, and visualization of protein/molecular data fields as referenced by their `@id`.

This notebook can serve as a template for further, more detailed analyses of FAIR²-compliant datasets via `mlcroissant` and Python.